# 04 · A neural network from scratch (backprop in NumPy)

**Goal:** derive backpropagation by hand, implement it, and verify it with gradient checking. Then train on a non-linear problem.

Network: $x\to z_1=xW_1+b_1\to a_1=\tanh(z_1)\to z_2=a_1W_2+b_2\to p=\sigma(z_2)$ with mean binary cross-entropy $L$.

Backward pass (batch size $n$, $\odot$ = elementwise):

$$\delta_2=\frac{p-y}{n},\quad \frac{\partial L}{\partial W_2}=a_1^\top\delta_2,\quad \frac{\partial L}{\partial b_2}=\textstyle\sum\delta_2,$$
$$\delta_1=(\delta_2W_2^\top)\odot(1-a_1^{2}),\quad \frac{\partial L}{\partial W_1}=x^\top\delta_1,\quad \frac{\partial L}{\partial b_1}=\textstyle\sum\delta_1 .$$

The pattern ("error signal times the transpose of what fed the layer") is the chain rule applied to matrices.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(3)

def make_moons(n, noise, rng):
    t = rng.uniform(0, np.pi, n)
    a = np.c_[np.cos(t), np.sin(t)]
    b = np.c_[1 - np.cos(t), 0.5 - np.sin(t)]
    X = np.r_[a, b] + noise * rng.normal(size=(2 * n, 2))
    y = np.r_[np.zeros(n), np.ones(n)]
    return X, y[:, None]

def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-z))

def init(H, rng):
    return {
        "W1": rng.normal(size=(2, H)) / np.sqrt(2), "b1": np.zeros(H),
        "W2": rng.normal(size=(H, 1)) / np.sqrt(H), "b2": np.zeros(1),
    }

def forward(P, X):
    z1 = X @ P["W1"] + P["b1"]
    a1 = np.tanh(z1)
    p = sigmoid(a1 @ P["W2"] + P["b2"])
    return a1, p

def loss(P, X, y):
    _, p = forward(P, X)
    eps = 1e-12
    return -np.mean(y * np.log(p + eps) + (1 - y) * np.log(1 - p + eps))

def backward(P, X, y):
    n = len(X)
    a1, p = forward(P, X)
    d2 = (p - y) / n
    d1 = (d2 @ P["W2"].T) * (1 - a1**2)
    return {"W2": a1.T @ d2, "b2": d2.sum(0), "W1": X.T @ d1, "b1": d1.sum(0)}

## Gradient check every parameter

Perturb each parameter entry, recompute the loss, and compare. Do this on a tiny network and a handful of points.

In [ ]:
Xs, ys = make_moons(5, 0.1, rng)
Ps = init(4, rng)
G = backward(Ps, Xs, ys)

worst = 0.0
h = 1e-6
for k in Ps:
    it = np.nditer(Ps[k], flags=["multi_index"])
    for _ in it:
        idx = it.multi_index
        old = Ps[k][idx]
        Ps[k][idx] = old + h; lp = loss(Ps, Xs, ys)
        Ps[k][idx] = old - h; lm = loss(Ps, Xs, ys)
        Ps[k][idx] = old
        num = (lp - lm) / (2 * h)
        worst = max(worst, abs(num - G[k][idx]))
print("largest abs difference:", worst)
assert worst < 1e-7

## Train on the two-moons problem (a linear model cannot solve this)

In [ ]:
X, y = make_moons(200, 0.15, rng)
Xte, yte = make_moons(200, 0.15, rng)

P = init(16, rng)
lr = 0.5
hist = []
for epoch in range(3000):
    G = backward(P, X, y)
    for k in P:
        P[k] -= lr * G[k]
    if epoch % 50 == 0:
        hist.append(loss(P, X, y))

acc_tr = np.mean((forward(P, X)[1] > 0.5) == y)
acc_te = np.mean((forward(P, Xte)[1] > 0.5) == yte)
print(f"train acc {acc_tr:.3f}   test acc {acc_te:.3f}   final loss {hist[-1]:.4f}")
assert acc_te > 0.9

gx, gy = np.meshgrid(np.linspace(-1.8, 2.8, 200), np.linspace(-1.3, 1.8, 200))
grid = np.c_[gx.ravel(), gy.ravel()]
zz = forward(P, grid)[1].reshape(gx.shape)

fig, axs = plt.subplots(1, 2, figsize=(10, 4))
axs[0].plot(np.arange(len(hist)) * 50, hist); axs[0].set_title("training loss")
axs[1].contourf(gx, gy, zz, levels=20, cmap="RdBu_r", alpha=0.6)
axs[1].scatter(X[:, 0], X[:, 1], c=y.ravel(), s=6, cmap="coolwarm", edgecolor="k", linewidth=0.2)
axs[1].set_title("decision surface")
plt.show()

## Exercises

1. Replace $\tanh$ with ReLU. What changes in $\delta_1$? Re-run the gradient check (watch for kinks at 0).
2. Add a second hidden layer by generalizing `forward`/`backward` into a loop over layers.
3. Implement momentum and Adam, and compare their loss curves to plain GD.
4. Skim a PyTorch tutorial and reproduce this exact model with `autograd`; check your hand-written gradients against `loss.backward()`.